# Allscripts SCM - Note Hydration

Populates `_exponent.omop_scm.note` from Allscripts SCM clinical notes.

## Source Tables
- `_exponent._bronze_allscripts_scm.dbo_scaobservation` - note section text (`ObsText`), keyed on `VisitID` + `DocumentID`
- `_exponent._bronze_allscripts_scm.dbo_scadocument` - document metadata: `VisitID`, `DocumentID`, `DocumentDimID`, `AuthoredDtm`, `IsActive`
- `_exponent._bronze_allscripts_scm.dbo_scadocumentdim` - document classification: `PatCareDocName`, `EntryType`, `Description`
- `_exponent._bronze_allscripts_scm.dbo_scaobscatalogdim` - note section/catalog labels for structured note text
- `_exponent._bronze_allscripts_scm_prod_01.dbo_cv3clientvisit` - visit bridge from `scadocument.VisitID` to patient `ClientGUID`

## Patient Join
- `dbo_scadocument.PatientDimID` is not a reliable bridge to `cv3client.GUID` for notes.
- Notes resolve person through `scadocument.VisitID -> cv3clientvisit.VisitIDCode -> cv3clientvisit.ClientGUID -> source_to_person`.
- Note: `source_to_person` for `allscripts_scm` must be populated (SCM person notebook must run first).

## Grain
- SCM structured notes are stored as many `scaobservation` rows per document.
- This notebook aggregates those rows into one OMOP note per `VisitID + DocumentID`.

## Pipeline
1. `silver_note` - staged temp view, full OMOP field set
2. MERGE to `omop_silver.note`
3. INSERT to `omop_mapping.source_to_note`
4. `gold` - resolves surrogate IDs and FK references
5. MERGE to `omop_scm.note`

## Filters (from client reference script)
- `d.IsActive = 1`
- `o.ObsText IS NOT NULL`
- `docdim.EntryType IN ('Free Text', 'Structured Note') OR o.ObsCatalogDimID = 430`
- Aggregated `note_text` must be at least 10 non-whitespace characters


In [0]:
%sql
-- TRUNCATE TABLE _exponent.omop_allscipts.note;

In [0]:
%sql
-- TRUNCATE TABLE _exponent.omop_scm.note;


In [0]:
%sql
-- DELETE FROM _exponent.omop_silver.note
-- WHERE source_system = 'allscripts_scm';


In [0]:
%sql
-- DELETE FROM _exponent.omop_mapping.source_to_note
-- WHERE source_system = 'allscripts_scm';


In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW silver_note AS
WITH cv3_visit_bridge AS (
  SELECT
    VisitIDCode,
    GUID,
    ClientGUID,
    ROW_NUMBER() OVER (
      PARTITION BY CAST(VisitIDCode AS STRING)
      ORDER BY Active DESC, AdmitDtm DESC, GUID DESC
    ) AS rn
  FROM _exponent._bronze_allscripts_scm_prod_01.dbo_cv3clientvisit
  WHERE VisitIDCode IS NOT NULL
    AND ClientGUID IS NOT NULL
    AND GUID IS NOT NULL
),

note_rows AS (
  SELECT
    o.VisitID,
    o.DocumentID,
    o.ObservationID,
    o.ObsDtm,
    d.DocumentDimID,
    docdim.PatCareDocName,
    docdim.Description AS document_description,
    obsdim.Description AS section_description,
    obsdim.ObsItemName AS section_name,
    cv.GUID AS visit_guid,
    stp.person_id,
    REGEXP_REPLACE(o.ObsText, '[\x00-\x1F\x7F]', ' ') AS obs_text
  FROM _exponent._bronze_allscripts_scm.dbo_scaobservation o
  JOIN _exponent._bronze_allscripts_scm.dbo_scadocument d
    ON d.VisitID    = o.VisitID
   AND d.DocumentID = o.DocumentID
   AND d.IsActive   = 1
  JOIN _exponent._bronze_allscripts_scm.dbo_scadocumentdim docdim
    ON docdim.DocumentDimID = d.DocumentDimID
  LEFT JOIN _exponent._bronze_allscripts_scm.dbo_scaobscatalogdim obsdim
    ON obsdim.ObsCatalogDimID = o.ObsCatalogDimID
  JOIN cv3_visit_bridge cv
    ON CAST(cv.VisitIDCode AS STRING) = CAST(d.VisitID AS STRING)
   AND cv.rn = 1
  JOIN _exponent.omop_mapping.source_to_person stp
    ON stp.person_source_value = CONCAT_WS(
         CHR(31),
         'allscripts_scm',
         'cv3client',
         'GUID',
         CAST(cv.ClientGUID AS STRING)
       )
   AND stp.active_flag = TRUE
  WHERE o.ObsText IS NOT NULL
    AND o.ObsDtm IS NOT NULL
    AND o.ObsDtm >= TIMESTAMP('1900-01-01')
    AND o.ObsDtm <= CURRENT_TIMESTAMP()
    AND (
      docdim.EntryType IN ('Free Text', 'Structured Note')
      OR o.ObsCatalogDimID = 430
    )
),

aggregated_notes AS (
SELECT
  CONCAT_WS(
    CHR(31),
    'allscripts_scm',
    'dbo_scadocument',
    'visitid',
    CAST(VisitID AS BIGINT),
    'documentid',
    CAST(DocumentID AS BIGINT)
  ) AS note_source_value,

  person_id,

  CAST(MIN(ObsDtm) AS DATE) AS note_date,
  MIN(ObsDtm) AS note_datetime,

  32817 AS note_type_concept_id,   -- EHR record
  3030653 AS note_class_concept_id, -- Clinical Note

  COALESCE(MAX(CONCAT(PatCareDocName, ' - ', document_description)), MAX(PatCareDocName), MAX(document_description), 'SCM Note') AS note_title,

  CONCAT_WS(
    '\n\n',
    TRANSFORM(
      SORT_ARRAY(
        COLLECT_LIST(
          NAMED_STRUCT(
            'sort_dt', ObsDtm,
            'sort_id', CAST(ObservationID AS BIGINT),
            'txt', CONCAT(COALESCE(section_description, section_name, 'Note Section'), ': ', obs_text)
          )
        )
      ),
      x -> x.txt
    )
  ) AS note_text,

  32678 AS encoding_concept_id, -- UTF-8
  4180186 AS language_concept_id, -- English

  NULL AS provider_id,

  CONCAT_WS(
    CHR(31),
    'allscripts_scm',
    'dbo_cv3clientvisit',
    'GUID',
    CAST(visit_guid AS STRING)
  ) AS visit_occurrence_source_value,

  NULL AS visit_detail_id,
  NULL AS note_event_id,
  NULL AS note_event_field_concept_id,

  'allscripts_scm' AS source_system
FROM note_rows
GROUP BY
  VisitID,
  DocumentID,
  person_id,
  visit_guid
)

SELECT *
FROM aggregated_notes
WHERE note_text IS NOT NULL
  AND LENGTH(TRIM(note_text)) >= 10;


In [0]:
%sql
MERGE INTO _exponent.omop_silver.note AS target
USING (
  SELECT *
  FROM (
    SELECT
      *,
      ROW_NUMBER() OVER (
        PARTITION BY note_source_value
        ORDER BY note_date DESC
      ) AS rn
    FROM silver_note
  )
  WHERE rn = 1
) AS source
ON target.note_source_value = source.note_source_value

WHEN MATCHED AND NOT (
     target.person_id                     <=> source.person_id
 AND target.note_date                     <=> source.note_date
 AND target.note_datetime                 <=> source.note_datetime
 AND target.note_type_concept_id          <=> source.note_type_concept_id
 AND target.note_class_concept_id         <=> source.note_class_concept_id
 AND target.note_title                    <=> source.note_title
 AND target.note_text                     <=> source.note_text
 AND target.encoding_concept_id           <=> source.encoding_concept_id
 AND target.language_concept_id           <=> source.language_concept_id
 AND target.provider_id                   <=> source.provider_id
 AND target.visit_occurrence_source_value <=> source.visit_occurrence_source_value
 AND target.visit_detail_id               <=> source.visit_detail_id
 AND target.note_event_id                <=> source.note_event_id
 AND target.note_event_field_concept_id  <=> source.note_event_field_concept_id
 AND target.source_system                 <=> source.source_system
) THEN UPDATE SET
  target.person_id                     = source.person_id,
  target.note_date                     = source.note_date,
  target.note_datetime                 = source.note_datetime,
  target.note_type_concept_id          = source.note_type_concept_id,
  target.note_class_concept_id         = source.note_class_concept_id,
  target.note_title                    = source.note_title,
  target.note_text                     = source.note_text,
  target.encoding_concept_id           = source.encoding_concept_id,
  target.language_concept_id           = source.language_concept_id,
  target.provider_id                   = source.provider_id,
  target.visit_occurrence_source_value = source.visit_occurrence_source_value,
  target.visit_detail_id               = source.visit_detail_id,
  target.note_event_id                 = source.note_event_id,
  target.note_event_field_concept_id   = source.note_event_field_concept_id,
  target.source_system                 = source.source_system,
  target.last_mod_tsp                  = CURRENT_TIMESTAMP()

WHEN NOT MATCHED THEN INSERT (
  note_source_value,
  person_id,
  note_date,
  note_datetime,
  note_type_concept_id,
  note_class_concept_id,
  note_title,
  note_text,
  encoding_concept_id,
  language_concept_id,
  provider_id,
  visit_occurrence_source_value,
  visit_detail_id,
  note_event_id,
  note_event_field_concept_id,
  source_system,
  last_mod_tsp
) VALUES (
  source.note_source_value,
  source.person_id,
  source.note_date,
  source.note_datetime,
  source.note_type_concept_id,
  source.note_class_concept_id,
  source.note_title,
  source.note_text,
  source.encoding_concept_id,
  source.language_concept_id,
  source.provider_id,
  source.visit_occurrence_source_value,
  source.visit_detail_id,
  source.note_event_id,
  source.note_event_field_concept_id,
  source.source_system,
  CURRENT_TIMESTAMP()
);

In [0]:
%sql
INSERT INTO _exponent.omop_mapping.source_to_note (
    source_system,
    note_source_value,
    active_flag,
    created_tsp,
    last_mod_tsp,
    merge_id,
    merge_reason
)
SELECT
    s.source_system,
    s.note_source_value,
    TRUE                AS active_flag,
    CURRENT_TIMESTAMP() AS created_tsp,
    CURRENT_TIMESTAMP() AS last_mod_tsp,
    NULL                AS merge_id,
    NULL                AS merge_reason
FROM (
    SELECT DISTINCT
        source_system,
        note_source_value
    FROM _exponent.omop_silver.note
    WHERE note_source_value IS NOT NULL
      AND source_system = 'allscripts_scm'
) s
LEFT ANTI JOIN _exponent.omop_mapping.source_to_note x
  ON s.note_source_value = x.note_source_value
 AND x.source_system = 'allscripts_scm';

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW gold AS
SELECT
  stn.note_id,
  n.person_id,
  n.note_date,
  n.note_datetime,
  n.note_type_concept_id,
  n.note_class_concept_id,
  n.note_title,
  n.note_text,
  n.encoding_concept_id,
  n.language_concept_id,
  n.provider_id,
  stvo.visit_occurrence_id,
  n.visit_detail_id,
  n.note_event_id,
  n.note_event_field_concept_id,
  n.note_source_value

FROM _exponent.omop_silver.note n
JOIN _exponent.omop_scm.person p
  ON p.person_id = n.person_id
JOIN _exponent.omop_mapping.source_to_note stn
  ON n.note_source_value = stn.note_source_value
 AND stn.source_system   = 'allscripts_scm'
 AND stn.active_flag     = TRUE
LEFT JOIN _exponent.omop_mapping.source_to_visit_occurrence stvo
  ON n.visit_occurrence_source_value = stvo.visit_occurrence_source_value
 AND stvo.source_system = 'allscripts_scm'
 AND stvo.active_flag   = TRUE
LEFT JOIN _exponent.omop_scm.death d
  ON d.person_id = n.person_id
WHERE n.source_system = 'allscripts_scm'
  AND (d.death_date IS NULL OR n.note_date <= d.death_date);

In [0]:
%sql
MERGE INTO _exponent.omop_scm.note AS target
USING gold AS source
ON target.note_id = source.note_id

WHEN MATCHED AND NOT (
     target.person_id             <=> source.person_id
 AND target.note_date             <=> source.note_date
 AND target.note_datetime         <=> source.note_datetime
 AND target.note_type_concept_id  <=> source.note_type_concept_id
 AND target.note_class_concept_id <=> source.note_class_concept_id
 AND target.note_title            <=> source.note_title
 AND target.note_text             <=> source.note_text
 AND target.encoding_concept_id   <=> source.encoding_concept_id
 AND target.language_concept_id   <=> source.language_concept_id
 AND target.provider_id           <=> source.provider_id
 AND target.visit_occurrence_id   <=> source.visit_occurrence_id
 AND target.visit_detail_id             <=> source.visit_detail_id
 AND target.note_event_id               <=> source.note_event_id
 AND target.note_event_field_concept_id <=> source.note_event_field_concept_id
 AND target.note_source_value     <=> source.note_source_value
) THEN UPDATE SET
  target.person_id             = source.person_id,
  target.note_date             = source.note_date,
  target.note_datetime         = source.note_datetime,
  target.note_type_concept_id  = source.note_type_concept_id,
  target.note_class_concept_id = source.note_class_concept_id,
  target.note_title            = source.note_title,
  target.note_text             = source.note_text,
  target.encoding_concept_id   = source.encoding_concept_id,
  target.language_concept_id   = source.language_concept_id,
  target.provider_id           = source.provider_id,
  target.visit_occurrence_id   = source.visit_occurrence_id,
  target.visit_detail_id             = source.visit_detail_id,
  target.note_event_id               = source.note_event_id,
  target.note_event_field_concept_id = source.note_event_field_concept_id,
  target.note_source_value     = source.note_source_value

WHEN NOT MATCHED THEN INSERT (
  note_id,
  person_id,
  note_date,
  note_datetime,
  note_type_concept_id,
  note_class_concept_id,
  note_title,
  note_text,
  encoding_concept_id,
  language_concept_id,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  note_event_id,
  note_event_field_concept_id,
  note_source_value
) VALUES (
  source.note_id,
  source.person_id,
  source.note_date,
  source.note_datetime,
  source.note_type_concept_id,
  source.note_class_concept_id,
  source.note_title,
  source.note_text,
  source.encoding_concept_id,
  source.language_concept_id,
  source.provider_id,
  source.visit_occurrence_id,
  source.visit_detail_id,
  source.note_event_id,
  source.note_event_field_concept_id,
  source.note_source_value
);

In [0]:
%sql
MERGE INTO _exponent.omop_allscripts.note AS target
USING gold AS source
ON target.note_id = source.note_id

WHEN MATCHED AND NOT (
     target.person_id             <=> source.person_id
 AND target.note_date             <=> source.note_date
 AND target.note_datetime         <=> source.note_datetime
 AND target.note_type_concept_id  <=> source.note_type_concept_id
 AND target.note_class_concept_id <=> source.note_class_concept_id
 AND target.note_title            <=> source.note_title
 AND target.note_text             <=> source.note_text
 AND target.encoding_concept_id   <=> source.encoding_concept_id
 AND target.language_concept_id   <=> source.language_concept_id
 AND target.provider_id           <=> source.provider_id
 AND target.visit_occurrence_id   <=> source.visit_occurrence_id
 AND target.visit_detail_id             <=> source.visit_detail_id
 AND target.note_event_id               <=> source.note_event_id
 AND target.note_event_field_concept_id <=> source.note_event_field_concept_id
 AND target.note_source_value     <=> source.note_source_value
) THEN UPDATE SET
  target.person_id             = source.person_id,
  target.note_date             = source.note_date,
  target.note_datetime         = source.note_datetime,
  target.note_type_concept_id  = source.note_type_concept_id,
  target.note_class_concept_id = source.note_class_concept_id,
  target.note_title            = source.note_title,
  target.note_text             = source.note_text,
  target.encoding_concept_id   = source.encoding_concept_id,
  target.language_concept_id   = source.language_concept_id,
  target.provider_id           = source.provider_id,
  target.visit_occurrence_id   = source.visit_occurrence_id,
  target.visit_detail_id             = source.visit_detail_id,
  target.note_event_id               = source.note_event_id,
  target.note_event_field_concept_id = source.note_event_field_concept_id,
  target.note_source_value     = source.note_source_value

WHEN NOT MATCHED THEN INSERT (
  note_id,
  person_id,
  note_date,
  note_datetime,
  note_type_concept_id,
  note_class_concept_id,
  note_title,
  note_text,
  encoding_concept_id,
  language_concept_id,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  note_event_id,
  note_event_field_concept_id,
  note_source_value
) VALUES (
  source.note_id,
  source.person_id,
  source.note_date,
  source.note_datetime,
  source.note_type_concept_id,
  source.note_class_concept_id,
  source.note_title,
  source.note_text,
  source.encoding_concept_id,
  source.language_concept_id,
  source.provider_id,
  source.visit_occurrence_id,
  source.visit_detail_id,
  source.note_event_id,
  source.note_event_field_concept_id,
  source.note_source_value
);

## QA
Run this after the SCM note notebook finishes. Expected result: large document-level note count, no missing `person_id`, high visit linkage, and non-empty note text.


In [ ]:
%sql
-- QA: SCM note load verification
SELECT
  COUNT(*) AS total_notes,
  COUNT(DISTINCT note_source_value) AS distinct_note_source_values,
  SUM(CASE WHEN person_id IS NULL THEN 1 ELSE 0 END) AS rows_missing_person_id,
  SUM(CASE WHEN visit_occurrence_id IS NULL THEN 1 ELSE 0 END) AS rows_missing_visit_occurrence_id,
  ROUND(100.0 * SUM(CASE WHEN visit_occurrence_id IS NOT NULL THEN 1 ELSE 0 END) / NULLIF(COUNT(*), 0), 2) AS pct_visit_linked,
  SUM(CASE WHEN note_text IS NULL OR LENGTH(TRIM(note_text)) = 0 THEN 1 ELSE 0 END) AS rows_missing_note_text,
  MIN(note_date) AS min_note_date,
  MAX(note_date) AS max_note_date
FROM _exponent.omop_scm.note;
